In [1]:
pip install mtcnn opencv-python tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.0 MB/s eta 0:00:00


In [ ]:
# use this when you have photos of full length

import os
import cv2
from mtcnn import MTCNN

# 1. Initialize the face detector
detector = MTCNN()

# 2. Define our paths
source_dir = '/content/drive/MyDrive/Face_Dataset'
dest_dir = '/content/drive/MyDrive/Cropped_Faces'

# Create the destination folder if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

print("Starting the face extraction process...\n")

# 3. Loop through every folder in your dataset
for person_name in os.listdir(source_dir):
    person_dir = os.path.join(source_dir, person_name)

    if not os.path.isdir(person_dir):
        continue

    dest_person_dir = os.path.join(dest_dir, person_name)
    os.makedirs(dest_person_dir, exist_ok=True)

    print(f"📷 Processing faces for: {person_name}...")

    success_count = 0
    fail_count = 0

    # 4. Loop through every image for that person
    for img_name in os.listdir(person_dir):
        img_path = os.path.join(person_dir, img_name)

        # Read the image
        img = cv2.imread(img_path)
        if img is None:
            fail_count += 1
            continue

        # Convert to RGB for MTCNN
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Detect faces
        results = detector.detect_faces(img_rgb)

        if results:
            # Get coordinates of the most prominent face
            x, y, w, h = results[0]['box']

            # Ensure coordinates are within bounds
            x, y = max(0, x), max(0, y)

            # Crop and save
            face_crop = img[y:y+h, x:x+w]
            dest_path = os.path.join(dest_person_dir, img_name)
            cv2.imwrite(dest_path, face_crop)
            success_count += 1
        else:
            fail_count += 1

    print(f"✅ Extracted {success_count} faces.")
    if fail_count > 0:
        print(f"⚠️ Skipped {fail_count} images (No face detected or unreadable file).")

print("\n🎉 All done! Check your Google Drive for the 'Cropped_Faces' folder.")

Starting the face extraction process...

❌ ERROR: Missing required folders in Face_Dataset: ['BomanIrani']
Please make sure you have both 'Prajwal' and 'BomanIrani' folders created before running this.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import pickle
import os

# ==========================================
# 1. Load the Cropped Dataset
# ==========================================
dataset_path = '/content/drive/MyDrive/Cropped_Faces'

print("Loading cropped dataset...")
# Load training data (80%)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224), # MobileNetV2 expects 224x224
    batch_size=16
)

# Load validation data (20%)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=16
)

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")

if num_classes < 2:
    print("❌ ERROR: You need at least 2 folders (e.g., 'Prajwal' and 'BomanIrani') to train the model!")
else:
    # Save the labels dictionary for your Gradio App later
    label_dict = {i: name for i, name in enumerate(class_names)}
    with open('/content/drive/MyDrive/labels.pkl', 'wb') as f:
        pickle.dump(label_dict, f)
    print("Labels saved to Drive!")

    # Performance optimization
    AUTOTUNE = tf.data.AUTOTUNE
    train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
    val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

    # ==========================================
    # 2. Build the Transfer Learning Model
    # ==========================================
    print("Building MobileNetV2 Model...")

    # Data Augmentation: Flips and slight rotations to prevent overfitting
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.1),
    ])

    # Load MobileNetV2 (Pre-trained on ImageNet)
    # We set include_top=False because we want to add our own classification layer
    base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

    # Freeze the base model so we don't destroy its pre-trained knowledge
    base_model.trainable = False

    # Create the new model pipeline
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x) # Scale pixels to [-1, 1]
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x) # 50% dropout to force the network to be robust
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)

    # Compile the model
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # ==========================================
    # 3. Train the Model
    # ==========================================
    print("Starting training...")

    # Stop early if the model stops improving
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=30,
        callbacks=[early_stop]
    )

# ==========================================
    # 4. Save the Final Model
    # ==========================================
    model_save_path = '/content/drive/MyDrive/face_model_v2.keras'
    model.save(model_save_path)
    print(f"✅ Training complete! Model saved to: {model_save_path}")

Loading cropped dataset...
Found 488 files belonging to 3 classes.
Using 391 files for training.
Found 488 files belonging to 3 classes.
Using 97 files for validation.
Found 3 classes: ['BomanIrani', 'Dhanraj', 'Prajwal']
Labels saved to Drive!
Building MobileNetV2 Model...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Starting training...
Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 151s 6s/step - accuracy: 0.7698 - loss: 0.5868 - val_accuracy: 0.9691 - val_loss: 0.1374
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 12s 504ms/step - accuracy: 0.9693 - loss: 0.1512 - val_accuracy: 0.9691 - val_loss: 0.0913
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 20s 483ms/step - accuracy: 0.9616 - loss: 0.1357 - val_accuracy: 0.9794 - val_loss: 0.0733
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 12s 508ms/step - accuracy: 0.9770 - loss: 0.0810 - val_accuracy: 0.9794 - val_loss: 0.0693
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 13s 528ms/step - accuracy: 0.9795 - loss: 0.0788 - val_accuracy: 0.9794 - val_loss: 0.0628
Epoch 6/30
2

In [3]:
import gradio as gr
import tensorflow as tf
import cv2
import numpy as np
import pickle
from mtcnn import MTCNN

# ==========================================
# 1. Load the Multi-Class Model
# ==========================================
print("Loading your updated 3-Person Model...")
model = tf.keras.models.load_model('/content/drive/MyDrive/face_model_v2.keras')

# Load the dictionary. It is already perfectly formatted as {ID: Name}
with open('/content/drive/MyDrive/labels.pkl', 'rb') as f:
    label_map = pickle.load(f)

detector = MTCNN()

# ==========================================
# 2. Define Authorized Users
# ==========================================
AUTHORIZED_USERS = ["Prajwal", "Dhanraj"]

# ==========================================
# 3. Prediction Logic
# ==========================================
def identify_friends(image):
    if image is None:
        return "Please upload an image or use the webcam."

    results = detector.detect_faces(image)

    if not results:
        return "❌ Scan Failed: No face detected."

    x, y, w, h = results[0]['box']
    x, y = max(0, x), max(0, y)

    face_crop = image[y:y+h, x:x+w]
    face_resized = cv2.resize(face_crop, (224, 224))

    # Preprocess for the CNN
    face_array = np.expand_dims(face_resized, axis=0).astype('float32')

    # Run the prediction
    predictions = model.predict(face_array)[0]
    predicted_id = np.argmax(predictions)
    confidence = predictions[predicted_id] * 100

    # Correctly grab the name using the ID integer
    predicted_name = label_map.get(predicted_id, "Unknown")

    # Grab diagnostics for debugging
    debug_stats = "\n\n--- AI Diagnostics ---\n"
    for i, prob in enumerate(predictions):
        name = label_map.get(i, "Unknown")
        debug_stats += f"{name}: {(prob * 100):.2f}%\n"

    # ==========================================
    # 4. The Multi-Person Rules
    # ==========================================
    if predicted_name in AUTHORIZED_USERS and confidence >= 85.0:
        return f"✅ ACCESS GRANTED.\nWelcome, {predicted_name}! (Confidence: {confidence:.1f}%)" + debug_stats

    elif predicted_name in AUTHORIZED_USERS and confidence < 85.0:
        return f"⚠️ SCAN INCONCLUSIVE.\nLooks like {predicted_name}, but confidence is too low ({confidence:.1f}%)." + debug_stats

    else:
        return f"🚫 ACCESS DENIED.\nSubject identified as: {predicted_name} ({confidence:.1f}% Match)" + debug_stats

# ==========================================
# 5. Launch the Interface
# ==========================================
demo = gr.Interface(
    fn=identify_friends,
    inputs=gr.Image(sources=["upload", "webcam"]),
    outputs=gr.Textbox(label="Security Output", text_align="center"),
    title="Prajwal System Access 🔐",
    description="Authorized personnel only. The AI will scan your face to verify your identity.",
    allow_flagging="never"
)

demo.launch(debug=True, share=True)

Loading your updated 3-Person Model...


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://de04c3c621848dc903.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://de04c3c621848dc903.gradio.live
